## Class Creation - Data Puller

In [1]:
# CLASS CREATION - STOCK_DATA_PULLER

from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame, TimeFrameUnit
from alpaca.data.enums import DataFeed
from datetime import datetime, timedelta
import pandas as pd
import numpy as np

class AlpacaDataPuller:
    """
    Pulls Stock Data from Alpaca API and establishes quartile-based baseline targets.
    """
    def __init__(
        self, 
        symbols: list[str], 
        api_key: str,
        secret_key: str,
        timeframe_amount: int = 5,
        timeframe_unit: str = "minute"
    ):
        self.symbols = symbols
        self.api_key = api_key
        self.secret_key = secret_key
        self.timeframe_amount = timeframe_amount
        self.timeframe_unit = timeframe_unit
        
        # Dictionary to store pulled self.dfs for different timeframes
        self.data = {}

        # Initialize Alpaca Client
        self.client = StockHistoricalDataClient(api_key, secret_key)

    def get_timeframe(self, amount=None, unit=None):
        if amount is None:
            amount = self.timeframe_amount
        if unit is None:
            unit = self.timeframe_unit

        unit = unit.lower()
        unit_map = {
            "minute": TimeFrameUnit.Minute,
            "min": TimeFrameUnit.Minute,
            "hour": TimeFrameUnit.Hour,
            "day": TimeFrameUnit.Day,
            "week": TimeFrameUnit.Week,
        }

        if unit not in unit_map:
            raise ValueError("timeframe_unit must be: minute, hour, day, or week")

        return TimeFrame(amount, unit_map[unit])

    def pull_symbol_data(self, symbol, start=None, end=None, timeframe_amount=None, timeframe_unit=None):
        if end is None:
            end_date = datetime.today()
        else:
            end_date = datetime(*end)

        if start is None:
            start_date = end_date - timedelta(days=365 * 10)
        else:
            start_date = datetime(*start)

        request = StockBarsRequest(
            symbol_or_symbols=symbol,
            timeframe=self.get_timeframe(timeframe_amount, timeframe_unit),
            start=start_date,
            end=end_date,
            feed=DataFeed.IEX  # Use IEX for free/paper subscriptions
        )

        bars = self.client.get_stock_bars(request)
        return bars.df

    def pull_symbols_data(self, symbols=None, start=None, end=None, timeframe_amount=None, timeframe_unit=None):
        if symbols is None:
            symbols = self.symbols

        data = {}
        for symbol in symbols:
            data[symbol] = self.pull_symbol_data(symbol, start, end, timeframe_amount, timeframe_unit)
        return data

    def pull_multi_timeframe_data(self, timeframes: dict, start=None, end=None):
        """
        Pulls data for multiple timeframes and stores it in self.data.
        timeframes dictionary format: {'daily': (1, 'day'), 'weekly': (1, 'week')}
        """
        for label, (amount, unit) in timeframes.items():
            print(f"Pulling data for timeframe: '{label}' ({amount} {unit})...")
            self.data[label] = self.pull_symbols_data(
                symbols=self.symbols,
                start=start,
                end=end,
                timeframe_amount=amount,
                timeframe_unit=unit
            )
        print("All timeframes pulled successfully!")

    # --- 5-CLASS QUARTILE BASELINE LOGIC ---

    def classify_return(self, r, q1, q3, thresh):
        if r < q1:
            return 0  # Strong Down
        elif r < -thresh:
            return 1  # Moderate Down
        elif r <= thresh:
            return 2  # Flat
        elif r <= q3:
            return 3  # Moderate Up
        else:
            return 4  # Strong Up

    def create_baselines(self, timeframe_label, zero_thresh=0.02, close_col='close'):
        """
        Establishes 5 classes based on quartiles and a zero threshold.
        Appends the 'Target' column to each symbol's DataFrame in self.data[timeframe_label].
        """
        if timeframe_label not in self.data:
            raise ValueError(f"Timeframe '{timeframe_label}' data has not been pulled yet.")
            
        print(f"\n=== 5-Class Baselines for Timeframe: {timeframe_label.upper()} ===")
        
        class_names = {
            0: "Strong Down (R < Q1)",
            1: "Moderate Down (Q1 <= R < -thresh)",
            2: "Flat (-thresh <= R <= thresh)",
            3: "Moderate Up (thresh < R <= Q3)",
            4: "Strong Up (R > Q3)"
        }
        
        for symbol, df in self.data[timeframe_label].items():
            df = df.copy()
            
            # Calculate next period's percentage return grouped by symbol if combined,
            # or just simple return if it is a single-ticker dataframe.
            if 'symbol' in df.columns:
                df['Next_Return'] = df.groupby('symbol')[close_col].transform(lambda x: x.pct_change().shift(-1))
            elif isinstance(df.index, pd.MultiIndex) and 'symbol' in df.index.names:
                df['Next_Return'] = df.groupby(level='symbol')[close_col].transform(lambda x: x.pct_change().shift(-1))
            else:
                df['Next_Return'] = df[close_col].pct_change().shift(-1)
            
            # Clean returns for calculating statistics
            clean_returns = df['Next_Return'].dropna()
            
            # Group and calculate targets per symbol to prevent target bleeding
            if 'symbol' in df.columns or (isinstance(df.index, pd.MultiIndex) and 'symbol' in df.index.names):
                def get_symbol_target(group):
                    g_clean = group['Next_Return'].dropna()
                    if len(g_clean) == 0:
                        group['Target'] = np.nan
                        return group
                    g_q1 = g_clean.quantile(0.25)
                    g_q3 = g_clean.quantile(0.75)
                    g_avg_mov = g_clean.abs().mean()
                    g_thresh = zero_thresh * g_avg_mov
                    
                    group['Target'] = group['Next_Return'].apply(
                        lambda x: self.classify_return(x, g_q1, g_q3, g_thresh) if pd.notna(x) else np.nan
                    )
                    return group
                
                if 'symbol' in df.columns:
                    df = df.groupby('symbol', group_keys=False).apply(get_symbol_target)
                else:
                    df = df.groupby(level='symbol', group_keys=False).apply(get_symbol_target)
            else:
                q1 = clean_returns.quantile(0.25)
                q3 = clean_returns.quantile(0.75)
                avg_movement = clean_returns.abs().mean()
                thresh = zero_thresh * avg_movement
                df['Target'] = df['Next_Return'].apply(
                    lambda x: self.classify_return(x, q1, q3, thresh) if pd.notna(x) else np.nan
                )
            
            # Drop the last row which contains NaN target
            df = df.dropna(subset=['Target'])
            df['Target'] = df['Target'].astype(int)
            
            # Store the updated DataFrame back in self.data
            self.data[timeframe_label][symbol] = df
            
            # Print class distribution per symbol
            if 'symbol' in df.columns:
                symbols_present = df['symbol'].unique()
            elif isinstance(df.index, pd.MultiIndex) and 'symbol' in df.index.names:
                symbols_present = df.index.get_level_values('symbol').unique()
            else:
                symbols_present = [symbol]
                
            for sym in symbols_present:
                if 'symbol' in df.columns:
                    sym_df = df[df['symbol'] == sym]
                elif isinstance(df.index, pd.MultiIndex) and 'symbol' in df.index.names:
                    sym_df = df.xs(sym, level='symbol', drop_level=False)
                else:
                    sym_df = df
                    
                sym_clean_returns = sym_df['Next_Return'].dropna()
                sym_q1 = sym_clean_returns.quantile(0.25)
                sym_q3 = sym_clean_returns.quantile(0.75)
                sym_avg = sym_clean_returns.abs().mean()
                sym_thresh = zero_thresh * sym_avg
                
                print(f"\nSymbol: {sym}")
                print(f"  Average Absolute Movement: {sym_avg * 100:.4f}%")
                print(f"  Flat Threshold (+/- {zero_thresh*100}% of Avg):  {sym_thresh * 100:.4f}%")
                print(f"  Q1 (25th percentile):             {sym_q1 * 100:.4f}%")
                print(f"  Q3 (75th percentile):             {sym_q3 * 100:.4f}%")
                
                counts = sym_df['Target'].value_counts().sort_index()
                total = len(sym_df)
                for c, count in counts.items():
                    print(f"  Class {c} | {class_names[c]:<35} : {count:<5} ({count/total*100:.2f}%)")


## SPY Example using AlpacaDataPuller

In [2]:
# Pull data for 3 representative companies and Index (SPY,MU, TSLA, GOOG)

MULTI_SYMBOL = ["MU", "GOOG", "TSLA", "SPY"]

API_KEY = "PKEN63LDIFPD43FZN5JQXXXHXV"
SECRET_KEY = "3pmEQxuC9R5vGL4y6YSfDMWq9fgSM8nBZae8QGHyoE3C"

# Initialize puller
puller = AlpacaDataPuller(MULTI_SYMBOL, API_KEY, SECRET_KEY)

# Define configurations to download (e.g. 1 hour, 1 day, 1 week bars)
timeframe_configs = {
    'hourly': (1, 'hour'),
    'daily': (1, 'day'),
    'weekly': (1, 'week')
}

# Pull 10 years of data (start dates will be limited by Alpaca for hourly)
puller.pull_multi_timeframe_data(timeframe_configs, start=(2016, 6, 20))

# Compute targets & prints metric summaries
puller.create_baselines('hourly')
puller.create_baselines('daily')
puller.create_baselines('weekly')


Pulling data for timeframe: 'hourly' (1 hour)...
Pulling data for timeframe: 'daily' (1 day)...
Pulling data for timeframe: 'weekly' (1 week)...
All timeframes pulled successfully!

=== 5-Class Baselines for Timeframe: HOURLY ===

Symbol: MU
  Average Absolute Movement: 0.7542%
  Flat Threshold (+/- 2.0% of Avg):  0.0151%
  Q1 (25th percentile):             -0.4505%
  Q3 (75th percentile):             0.4946%
  Class 0 | Strong Down (R < Q1)                : 2705  (25.00%)
  Class 1 | Moderate Down (Q1 <= R < -thresh)   : 2463  (22.77%)
  Class 2 | Flat (-thresh <= R <= thresh)       : 211   (1.95%)
  Class 3 | Moderate Up (thresh < R <= Q3)      : 2734  (25.27%)
  Class 4 | Strong Up (R > Q3)                  : 2705  (25.00%)

Symbol: GOOG
  Average Absolute Movement: 0.4737%
  Flat Threshold (+/- 2.0% of Avg):  0.0095%
  Q1 (25th percentile):             -0.2908%
  Q3 (75th percentile):             0.3142%
  Class 0 | Strong Down (R < Q1)                : 2665  (25.00%)
  Class 1 | M


    Different timeframes represent very different data distributions and market microstructures:
1. **The Volatility vs. Noise Trade-off**
    - **Hourly Data (High Noise, High Volume):**
        - Characteristics: You have a huge number of data points (44k+ rows), which ML models love. However, the price moves are very small (average absolute move is only 0.58%) and dominated by short-term noise.
        - ML Hypothesis: Simpler models (like Logistic Regression) might perform better here because they resist overfitting to noise, whereas complex tree models (like XGBoost) might overfit unless heavily regularized.
    
    - **Weekly Data (Low Noise, Low Volume): Characteristics: You have very clean signals and larger price moves (average absolute move is 4.74%), but very few data points (only ~500 rows).**
        - ML Hypothesis: Tree models might struggle here due to the lack of training data, but if they work, the trades are highly profitable.

    - **Daily Data (The Benchmark):**
        - Characteristics: The middle ground (average absolute move of 1.95%), with about 2,500 data points.

2. **By training separate models for the three timeframes, you can answer the following questions:**
    - **Which timeframe is the most "predictable"?**
        - Compare the test set classification accuracy of each model against its respective baseline (e.g. comparing the model's daily accuracy to the 25% random quartile baseline).

    - **Which model handles noise best?**
        - Does LightGBM/XGBoost outperform Logistic Regression on hourly data (where it can learn complex patterns from 44,000 rows) or on daily data?

    - **How do transaction costs impact performance?**
        - An hourly model might have 60% accuracy, but if it recommends trading 10 times a day, the broker fees and slippage will wipe out all profits. A weekly model trades less frequently and is much cheaper to execute in real life. You can show this comparison!

In [3]:
import pandas as pd

'''create simple dataframe tables
symbol: Multiindex
'''
# Concatenate individual DataFrames since puller now returns dictionary with separate symbol keys
hourly_data = pd.concat(puller.data['hourly'].values())
weekly_data = pd.concat(puller.data['weekly'].values())
daily_data  = pd.concat(puller.data['daily'].values())


In [4]:
hourly_data.reset_index(inplace=True)
weekly_data.reset_index(inplace=True)
daily_data.reset_index(inplace=True)

### Feature Engineering: The Weekly Warmup & Dynamic Lookback Problem

When using technical indicators in time-series data, we lose the first $N$ rows as a "warmup period" (e.g. a 200-period average requires 200 days/weeks of history before it can output its first value). 

* **Hourly / Daily**: 200 periods is highly feasible (only 10 months of history on daily, and less than 30 trading days on hourly). We only lose ~8% of daily rows.
* **Weekly**: 200 periods is **200 weeks (~4 years)** of data. With a 10-year dataset, using a 200-week lookback forces us to discard **40%** of our training rows.

#### Dynamic Lookback Strategy:
To solve this, our updated `feature_create` class accepts a `timeframe` argument in its constructor and dynamically scales down the rolling windows for weekly data (capping lookbacks at 30 weeks). This preserves valuable data while keeping the indicators stationary and representative.

In [5]:
class feature_create:
    '''Class to help incorporate all methods for 
    metric creation. Computes metrics grouped by symbol to prevent data leakage
    and dynamically scales lookback windows depending on the timeframe.'''

    def __init__(self, df, timeframe="daily"):
        self.df = df
        self.timeframe = timeframe.lower()
    
    def calculate_moving_avg(self):
        # Set windows based on timeframe (weekly is scaled down to prevent massive warmup data loss)
        if self.timeframe == "weekly":
            windows = (5, 10, 20, 30)
        else:
            windows = (10, 20, 50, 100, 200)
            
        # Group by symbol before rolling or ewm calculations to prevent cross-symbol bleeding
        for w in windows:
            self.df[f'SMA_{w}'] = self.df.groupby('symbol')['close'].transform(lambda x: x.rolling(window=w).mean())
            self.df[f'ewm_{w}'] = self.df.groupby('symbol')['close'].transform(lambda x: x.ewm(span=w, adjust=False).mean())
            self.df[f'price_to_ma{w}'] = self.df['close'] / self.df[f'SMA_{w}'] - 1
            
        # Moving average crossover
        if self.timeframe == "weekly":
            self.df['ma_cross'] = self.df['SMA_10'] / self.df['SMA_30'] - 1
        else:
            self.df['ma_cross'] = self.df['SMA_50'] / self.df['SMA_200'] - 1
        return self

    def bollinger(self):
        self.df['BB_Mid'] = self.df.groupby('symbol')['close'].transform(lambda x: x.rolling(window=20).mean())
        bb_std = self.df.groupby('symbol')['close'].transform(lambda x: x.rolling(window=20).std())
        self.df['BB_Upper'] = self.df['BB_Mid'] + 2 * bb_std
        self.df['BB_Lower'] = self.df['BB_Mid'] - 2 * bb_std
        
        # Bollinger Band Percentile (bb_pct - where price sits in the band: 0 = bottom, 1 = top)
        denom = self.df['BB_Upper'] - self.df['BB_Lower']
        self.df['bb_pct'] = np.where(denom == 0, 0.5, (self.df['close'] - self.df['BB_Lower']) / denom)
        return self

    def RSI(self):
        def compute_rsi(series):
            delta = series.diff()
            gain = delta.clip(lower=0)
            loss = -delta.clip(upper=0)
            avg_gain = gain.ewm(alpha=1/14, min_periods=14, adjust=False).mean()
            avg_loss = loss.ewm(alpha=1/14, min_periods=14, adjust=False).mean()
            rs = avg_gain / avg_loss
            return 100 - (100 / (1 + rs))
            
        self.df['RSI'] = self.df.groupby('symbol')['close'].transform(compute_rsi)
        return self

    def MACD(self):
        def compute_macd_line(series):
            return series.ewm(span=12, adjust=False).mean() - series.ewm(span=26, adjust=False).mean()
        def compute_macd_signal(macd_series):
            return macd_series.ewm(span=9, adjust=False).mean()
            
        # Divided by close to keep MACD stationary over long horizons
        macd_line = self.df.groupby('symbol')['close'].transform(compute_macd_line)
        self.df['MACD'] = macd_line / self.df['close']
        self.df['MACD_Signal'] = self.df.groupby('symbol')['MACD'].transform(compute_macd_signal)
        self.df['MACD_Hist'] = self.df['MACD'] - self.df['MACD_Signal']
        return self

    def ret(self):
        def ret_1(series):
            return series.pct_change()
        def log_ret(series):
            return np.log(series).diff()

        # daily returns (the core signal)
        self.df['ret_1'] = self.df.groupby('symbol')['close'].transform(ret_1)
        self.df['log_ret'] = self.df.groupby('symbol')['close'].transform(log_ret)
        
        # yesterday / 2 days ago / 3 days ago returns (short-term memory)
        for lag in (1, 2, 3):
            self.df[f"ret_lag{lag}"] = self.df.groupby('symbol')["ret_1"].transform(lambda x: x.shift(lag))
        return self

    def momentum(self):
        if self.timeframe == "weekly":
            horizons = (2, 4, 8, 12, 26)  # 2 weeks, 1 month, 2 months, 3 months, 6 months
        else:
            horizons = (10, 20, 30, 50, 100, 200)
            
        for k in horizons:
            self.df[f"mom_{k}"] = self.df.groupby('symbol')['close'].transform(lambda x: x / x.shift(k) - 1)
        return self

    def volatility_and_volume(self):
        # Realized volatility (rolling standard deviation of returns)
        vol_w = 10 if self.timeframe == "weekly" else 20
        self.df[f'volatility_{vol_w}'] = self.df.groupby('symbol')['ret_1'].transform(lambda x: x.rolling(window=vol_w).std())
        
        # Volume ratios
        vol_windows = (4, 10, 26) if self.timeframe == "weekly" else (10, 20, 50)
        for w in vol_windows:
            vol_avg = self.df.groupby('symbol')['volume'].transform(lambda x: x.rolling(window=w).mean())
            self.df[f'vol_ratio{w}'] = self.df['volume'] / vol_avg
        return self

    def advanced_indicators(self):
        # 1. Trading Range
        self.df['range_hl'] = (self.df['high'] - self.df['low']) / self.df['close']
        
        # 2. Overnight Gap
        prev_close = self.df.groupby('symbol')['close'].shift(1)
        self.df['gap'] = (self.df['open'] - prev_close) / prev_close
        
        # 3. Average True Range
        tr1 = self.df['high'] - self.df['low']
        tr2 = (self.df['high'] - prev_close).abs()
        tr3 = (self.df['low'] - prev_close).abs()
        tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
        
        alpha = 1/10 if self.timeframe == "weekly" else 1/14
        min_periods = 10 if self.timeframe == "weekly" else 14
        atr_raw = tr.groupby(self.df['symbol']).transform(lambda x: x.ewm(alpha=alpha, min_periods=min_periods, adjust=False).mean())
        self.df[f'atr_{int(1/alpha)}'] = atr_raw / self.df['close']
        
        # 4. Average Directional Index
        up_move = self.df.groupby('symbol')['high'].diff()
        down_move = -self.df.groupby('symbol')['low'].diff()
        plus_dm = pd.Series(np.where((up_move > down_move) & (up_move > 0), up_move, 0.0), index=self.df.index)
        minus_dm = pd.Series(np.where((down_move > up_move) & (down_move > 0), down_move, 0.0), index=self.df.index)
        
        plus_dm_smoothed = plus_dm.groupby(self.df['symbol']).transform(lambda x: x.ewm(alpha=alpha, min_periods=min_periods, adjust=False).mean())
        minus_dm_smoothed = minus_dm.groupby(self.df['symbol']).transform(lambda x: x.ewm(alpha=alpha, min_periods=min_periods, adjust=False).mean())
        
        plus_di = 100 * plus_dm_smoothed / atr_raw
        minus_di = 100 * minus_dm_smoothed / atr_raw
        dx = 100 * (plus_di - minus_di).abs() / (plus_di + minus_di)
        self.df[f'adx_{int(1/alpha)}'] = dx.groupby(self.df['symbol']).transform(lambda x: x.ewm(alpha=alpha, min_periods=min_periods, adjust=False).mean())
        
        # 5. Money Flow Index
        typical = (self.df['high'] + self.df['low'] + self.df['close']) / 3
        money_flow = typical * self.df['volume']
        typical_shift = typical.groupby(self.df['symbol']).shift(1)
        
        pos_flow = pd.Series(np.where(typical > typical_shift, money_flow, 0.0), index=self.df.index)
        neg_flow = pd.Series(np.where(typical < typical_shift, money_flow, 0.0), index=self.df.index)
        
        mfi_w = 10 if self.timeframe == "weekly" else 14
        pos_mf = pos_flow.groupby(self.df['symbol']).transform(lambda x: x.rolling(mfi_w).sum())
        neg_mf = neg_flow.groupby(self.df['symbol']).transform(lambda x: x.rolling(mfi_w).sum())
        self.df[f'mfi_{mfi_w}'] = 100 - (100 / (1 + pos_mf / neg_mf))
        
        # 6. Chaikin Money Flow
        denom = self.df['high'] - self.df['low']
        denom = np.where(denom == 0, 1e-9, denom)
        mfm = ((self.df['close'] - self.df['low']) - (self.df['high'] - self.df['close'])) / denom
        mfv = mfm * self.df['volume']
        
        cmf_w = 10 if self.timeframe == "weekly" else 20
        num_sum = mfv.groupby(self.df['symbol']).transform(lambda x: x.rolling(cmf_w).sum())
        denom_sum = self.df['volume'].groupby(self.df['symbol']).transform(lambda x: x.rolling(cmf_w).sum())
        self.df[f'cmf_{cmf_w}'] = num_sum / denom_sum
        return self

    def create_all_features(self):
        '''Runs all feature engineering methods on the DataFrame in one call.'''
        self.calculate_moving_avg()
        self.bollinger()
        self.RSI()
        self.MACD()
        self.ret()
        self.momentum()
        self.volatility_and_volume()
        self.advanced_indicators()
        return self


In [6]:
hourly_data_feature = feature_create(hourly_data, timeframe="hourly").create_all_features()
daily_data_feature = feature_create(daily_data, timeframe="daily").create_all_features()
weekly_data_feature = feature_create(weekly_data, timeframe="weekly").create_all_features()

# View one of the updated DataFrames with all columns!
daily_data_feature.df.head()


,symbol,timestamp,open,high,low,close,volume,trade_count,vwap,Next_Return,...,volatility_20,vol_ratio10,vol_ratio20,vol_ratio50,range_hl,gap,atr_14,adx_14,mfi_14,cmf_20
0,MU,2020-07-27 04:00:00+00:00,50.670,51.700,50.495,51.605,166507.0,1860.0,51.367901,-0.028389,...,NaN,NaN,NaN,NaN,0.023350,NaN,NaN,NaN,NaN,NaN
1,MU,2020-07-28 04:00:00+00:00,51.200,51.225,50.050,50.140,175657.0,1619.0,50.429399,0.004188,...,NaN,NaN,NaN,NaN,0.023434,-0.007848,NaN,NaN,NaN,NaN
2,MU,2020-07-29 04:00:00+00:00,50.265,50.615,49.805,50.350,163859.0,1823.0,50.204913,0.007349,...,NaN,NaN,NaN,NaN,0.016087,0.002493,NaN,NaN,NaN,NaN
3,MU,2020-07-30 04:00:00+00:00,49.750,50.745,49.110,50.720,313145.0,2201.0,49.813198,-0.014294,...,NaN,NaN,NaN,NaN,0.032236,-0.011917,NaN,NaN,NaN,NaN
4,MU,2020-07-31 04:00:00+00:00,50.650,50.670,49.270,49.995,277686.0,2331.0,49.930770,0.008001,...,NaN,NaN,NaN,NaN,0.028003,-0.001380,NaN,NaN,NaN,NaN


In [7]:
# 3. View number of NaN values per column for hourly data
hourly_data_feature.df.isna().sum()

symbol              0
timestamp           0
open                0
high                0
low                 0
close               0
volume              0
trade_count         0
vwap                0
Next_Return         0
Target              0
SMA_10             36
ewm_10              0
price_to_ma10      36
SMA_20             76
ewm_20              0
price_to_ma20      76
SMA_50            196
ewm_50              0
price_to_ma50     196
SMA_100           396
ewm_100             0
price_to_ma100    396
SMA_200           796
ewm_200             0
price_to_ma200    796
ma_cross          796
BB_Mid             76
BB_Upper           76
BB_Lower           76
bb_pct             76
RSI                56
MACD                0
MACD_Signal         0
MACD_Hist           0
ret_1               4
log_ret             4
ret_lag1            8
ret_lag2           12
ret_lag3           16
mom_10             40
mom_20             80
mom_30            120
mom_50            200
mom_100           400
mom_200   

In [8]:
daily_data_feature.df.isna().sum()

symbol              0
timestamp           0
open                0
high                0
low                 0
close               0
volume              0
trade_count         0
vwap                0
Next_Return         0
Target              0
SMA_10             36
ewm_10              0
price_to_ma10      36
SMA_20             76
ewm_20              0
price_to_ma20      76
SMA_50            196
ewm_50              0
price_to_ma50     196
SMA_100           396
ewm_100             0
price_to_ma100    396
SMA_200           796
ewm_200             0
price_to_ma200    796
ma_cross          796
BB_Mid             76
BB_Upper           76
BB_Lower           76
bb_pct             76
RSI                56
MACD                0
MACD_Signal         0
MACD_Hist           0
ret_1               4
log_ret             4
ret_lag1            8
ret_lag2           12
ret_lag3           16
mom_10             40
mom_20             80
mom_30            120
mom_50            200
mom_100           400
mom_200   

In [9]:
# View one of the updated DataFrames with the new columns
weekly_data_feature.df.isna().sum()

symbol             0
timestamp          0
open               0
high               0
low                0
close              0
volume             0
trade_count        0
vwap               0
Next_Return        0
Target             0
SMA_5             16
ewm_5              0
price_to_ma5      16
SMA_10            36
ewm_10             0
price_to_ma10     36
SMA_20            76
ewm_20             0
price_to_ma20     76
SMA_30           116
ewm_30             0
price_to_ma30    116
ma_cross         116
BB_Mid            76
BB_Upper          76
BB_Lower          76
bb_pct            76
RSI               56
MACD               0
MACD_Signal        0
MACD_Hist          0
ret_1              4
log_ret            4
ret_lag1           8
ret_lag2          12
ret_lag3          16
mom_2              8
mom_4             16
mom_8             32
mom_12            48
mom_26           104
volatility_10     40
vol_ratio4        12
vol_ratio10       36
vol_ratio26      100
range_hl           0
gap          

<H5> Now we can safely discard these rows on the weekly data as it is no longer a significant data loss! <H5>

In [10]:
#Replace infinities with NaN
hourly_data_feature.df = hourly_data_feature.df.replace([np.inf,-np.inf],np.nan)
daily_data_feature.df = daily_data_feature.df.replace([np.inf,-np.inf],np.nan)
weekly_data_feature.df = weekly_data_feature.df.replace([np.inf,-np.inf],np.nan)

#Drop NaN rows
hourly_data_feature.df.dropna(inplace = True)
daily_data_feature.df.dropna(inplace = True)
weekly_data_feature.df.dropna(inplace = True)

# View dataset again
hourly_data_feature.df.isna().sum()



symbol            0
timestamp         0
open              0
high              0
low               0
close             0
volume            0
trade_count       0
vwap              0
Next_Return       0
Target            0
SMA_10            0
ewm_10            0
price_to_ma10     0
SMA_20            0
ewm_20            0
price_to_ma20     0
SMA_50            0
ewm_50            0
price_to_ma50     0
SMA_100           0
ewm_100           0
price_to_ma100    0
SMA_200           0
ewm_200           0
price_to_ma200    0
ma_cross          0
BB_Mid            0
BB_Upper          0
BB_Lower          0
bb_pct            0
RSI               0
MACD              0
MACD_Signal       0
MACD_Hist         0
ret_1             0
log_ret           0
ret_lag1          0
ret_lag2          0
ret_lag3          0
mom_10            0
mom_20            0
mom_30            0
mom_50            0
mom_100           0
mom_200           0
volatility_20     0
vol_ratio10       0
vol_ratio20       0
vol_ratio50       0


In [11]:
# View dataset again
daily_data_feature.df.isna().sum()

symbol            0
timestamp         0
open              0
high              0
low               0
close             0
volume            0
trade_count       0
vwap              0
Next_Return       0
Target            0
SMA_10            0
ewm_10            0
price_to_ma10     0
SMA_20            0
ewm_20            0
price_to_ma20     0
SMA_50            0
ewm_50            0
price_to_ma50     0
SMA_100           0
ewm_100           0
price_to_ma100    0
SMA_200           0
ewm_200           0
price_to_ma200    0
ma_cross          0
BB_Mid            0
BB_Upper          0
BB_Lower          0
bb_pct            0
RSI               0
MACD              0
MACD_Signal       0
MACD_Hist         0
ret_1             0
log_ret           0
ret_lag1          0
ret_lag2          0
ret_lag3          0
mom_10            0
mom_20            0
mom_30            0
mom_50            0
mom_100           0
mom_200           0
volatility_20     0
vol_ratio10       0
vol_ratio20       0
vol_ratio50       0


In [12]:
# View dataset again
weekly_data_feature.df.isna().sum()

symbol           0
timestamp        0
open             0
high             0
low              0
close            0
volume           0
trade_count      0
vwap             0
Next_Return      0
Target           0
SMA_5            0
ewm_5            0
price_to_ma5     0
SMA_10           0
ewm_10           0
price_to_ma10    0
SMA_20           0
ewm_20           0
price_to_ma20    0
SMA_30           0
ewm_30           0
price_to_ma30    0
ma_cross         0
BB_Mid           0
BB_Upper         0
BB_Lower         0
bb_pct           0
RSI              0
MACD             0
MACD_Signal      0
MACD_Hist        0
ret_1            0
log_ret          0
ret_lag1         0
ret_lag2         0
ret_lag3         0
mom_2            0
mom_4            0
mom_8            0
mom_12           0
mom_26           0
volatility_10    0
vol_ratio4       0
vol_ratio10      0
vol_ratio26      0
range_hl         0
gap              0
atr_10           0
adx_10           0
mfi_10           0
cmf_10           0
dtype: int64

In [14]:
FEATURE_COLS = ["ret_1", "ret_lag1", "ret_lag2", "ret_lag3", "price_to_ma10", "price_to_ma50", "price_to_ma200", "ma_cross",
    "RSI", 
    "MACD_Hist",
    "adx_14",          # Note: Dynamically handles weekly data
    "volatility_20", # Note: Dynamically handles weekly data
    "vol_ratio20",
    "range_hl", 
    "gap"]

daily_data_feature.df[FEATURE_COLS].info()

<class 'pandas.DataFrame'>
Index: 5181 entries, 200 to 5980
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   ret_1           5181 non-null   float64
 1   ret_lag1        5181 non-null   float64
 2   ret_lag2        5181 non-null   float64
 3   ret_lag3        5181 non-null   float64
 4   price_to_ma10   5181 non-null   float64
 5   price_to_ma50   5181 non-null   float64
 6   price_to_ma200  5181 non-null   float64
 7   ma_cross        5181 non-null   float64
 8   RSI             5181 non-null   float64
 9   MACD_Hist       5181 non-null   float64
 10  adx_14          5181 non-null   float64
 11  volatility_20   5181 non-null   float64
 12  vol_ratio20     5181 non-null   float64
 13  range_hl        5181 non-null   float64
 14  gap             5181 non-null   float64
dtypes: float64(15)
memory usage: 647.6 KB


## 6. Data Preprocessing

At this point all three tables are ready to be preprocessed for ML algorithms

In [15]:
import tensorflow as tf
from sklearn.preprocessing import StandardScaler

def preprocess_and_split(table, zero_thresh=0.02):
    ''' Function to accept timeframe data - either hourly, daily or weekly
    and perform data preprocessing, standardization of features, chronological splitting,
    and target computation based strictly on training set return quantiles to prevent look-ahead bias.'''

    # Dynamically select features based on timeframe
    ma_cols = ["price_to_ma10", "price_to_ma50", "price_to_ma200"] if "price_to_ma50" in table.columns else ["price_to_ma5", "price_to_ma20"]
    vol_col = ["vol_ratio20"] if "vol_ratio20" in table.columns else ["vol_ratio10"]

    FEATURE_COLS = [
        "ret_1", "ret_lag1", "ret_lag2", "ret_lag3",
        "ma_cross", "RSI", "MACD_Hist", "range_hl", "gap",
        "volatility_10" if "volatility_10" in table.columns else "volatility_20",
        "adx_10" if "adx_10" in table.columns else "adx_14"
    ] + ma_cols + vol_col

    # Clean data (replace infs with NaN and drop NaNs in features and Next_Return)
    table = table.replace([np.inf, -np.inf], np.nan).dropna(subset=FEATURE_COLS + ["Next_Return"]).copy()
    
    # Sort chronologically by timestamp column
    table['timestamp'] = pd.to_datetime(table['timestamp'])
    table = table.sort_values('timestamp')
    
    # Compute unique sorted timestamps to find global cutoff dates
    unique_dates = sorted(table['timestamp'].unique())
    N = len(unique_dates)
    
    TRAIN_FRAC = 0.70
    VAL_FRAC   = 0.15
    
    TRAIN_END = unique_dates[int(N * TRAIN_FRAC)]
    VAL_END   = unique_dates[int(N * (TRAIN_FRAC + VAL_FRAC))]
    
    # Split data chronologically
    train_df = table[table['timestamp'] <= TRAIN_END].copy()
    val_df   = table[(table['timestamp'] > TRAIN_END) & (table['timestamp'] <= VAL_END)].copy()
    test_df  = table[table['timestamp'] > VAL_END].copy()

    # Re-calculate Target based only on training set return quantiles to prevent target leakage
    def classify_return(r, q1, q3, thresh):
        if r < q1:
            return 0  # Strong Down
        elif r < -thresh:
            return 1  # Moderate Down
        elif r <= thresh:
            return 2  # Flat
        elif r <= q3:
            return 3  # Moderate Up
        else:
            return 4  # Strong Up

    # We compute thresholds per symbol from train_df and apply to train, val, and test
    symbols = table['symbol'].unique() if 'symbol' in table.columns else [None]
    
    for df in [train_df, val_df, test_df]:
        df['Target'] = np.nan

    for sym in symbols:
        # Get train slice for this symbol
        if sym is not None:
            train_sym = train_df[train_df['symbol'] == sym]
        else:
            train_sym = train_df
            
        clean_returns = train_sym['Next_Return'].dropna()
        if len(clean_returns) == 0:
            continue
            
        q1 = clean_returns.quantile(0.25)
        q3 = clean_returns.quantile(0.75)
        avg_movement = clean_returns.abs().mean()
        thresh = zero_thresh * avg_movement
        
        # Apply classifier to all splits
        if sym is not None:
            train_df.loc[train_df['symbol'] == sym, 'Target'] = train_df.loc[train_df['symbol'] == sym, 'Next_Return'].apply(
                lambda x: classify_return(x, q1, q3, thresh) if pd.notna(x) else np.nan
            )
            val_df.loc[val_df['symbol'] == sym, 'Target'] = val_df.loc[val_df['symbol'] == sym, 'Next_Return'].apply(
                lambda x: classify_return(x, q1, q3, thresh) if pd.notna(x) else np.nan
            )
            test_df.loc[test_df['symbol'] == sym, 'Target'] = test_df.loc[test_df['symbol'] == sym, 'Next_Return'].apply(
                lambda x: classify_return(x, q1, q3, thresh) if pd.notna(x) else np.nan
            )
        else:
            train_df['Target'] = train_df['Next_Return'].apply(
                lambda x: classify_return(x, q1, q3, thresh) if pd.notna(x) else np.nan
            )
            val_df['Target'] = val_df['Next_Return'].apply(
                lambda x: classify_return(x, q1, q3, thresh) if pd.notna(x) else np.nan
            )
            test_df['Target'] = test_df['Next_Return'].apply(
                lambda x: classify_return(x, q1, q3, thresh) if pd.notna(x) else np.nan
            )

    # Drop any remaining rows where Target is NaN
    train_df = train_df.dropna(subset=['Target'])
    val_df = val_df.dropna(subset=['Target'])
    test_df = test_df.dropna(subset=['Target'])
    
    # Cast to int
    train_df['Target'] = train_df['Target'].astype(int)
    val_df['Target'] = val_df['Target'].astype(int)
    test_df['Target'] = test_df['Target'].astype(int)

    X_train = train_df[FEATURE_COLS].to_numpy()
    Y_train = train_df['Target'].to_numpy()
    
    X_val = val_df[FEATURE_COLS].to_numpy()
    Y_val = val_df['Target'].to_numpy()
    
    X_test = test_df[FEATURE_COLS].to_numpy()
    Y_test = test_df['Target'].to_numpy()

    print('Size of X_train', X_train.shape)
    print('Size of y_train', Y_train.shape)
    print('Size of X_val', X_val.shape)
    print('Size of y_val', Y_val.shape)
    print('Size of X_test', X_test.shape)
    print('Size of y_test', Y_test.shape)

    return X_train, Y_train, X_val, Y_val, X_test, Y_test


In [26]:
def mlearn_timeframe(time):

    if time.lower() == "hourly":
        X_train,Y_train,X_val,Y_val,X_test,Y_test = preprocess_and_split(hourly_data_feature.df)
        # Standardizing the features using scaler fit ONLY on training set
        sc_x = StandardScaler()
        X_train_std = sc_x.fit_transform(X_train)
        X_val_std = sc_x.transform(X_val)      
        X_test_std = sc_x.transform(X_test)
    
    elif time.lower() =='daily':
        X_train,Y_train,X_val,Y_val,X_test,Y_test = preprocess_and_split(daily_data_feature.df)
        # Standardizing the features using scaler fit ONLY on training set
        sc_x = StandardScaler()
        X_train_std = sc_x.fit_transform(X_train)
        X_val_std = sc_x.transform(X_val)      
        X_test_std = sc_x.transform(X_test)
    
    elif time.lower() =='weekly':
        X_train,Y_train,X_val,Y_val,X_test,Y_test = preprocess_and_split(weekly_data_feature.df)
        # Standardizing the features using scaler fit ONLY on training set
        sc_x = StandardScaler()
        X_train_std = sc_x.fit_transform(X_train)
        X_val_std = sc_x.transform(X_val)      
        X_test_std = sc_x.transform(X_test)
    
    else:
        raise NameError("Enter the correct name of the timeframe")
    
    from sklearn.linear_model import LogisticRegression
    from sklearn.dummy import DummyClassifier
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
    RANDOM_STATE = 42

    # model 1: majority-class baseline (the floor)
    majority = DummyClassifier(strategy="most_frequent").fit(X_train_std, Y_train)
    maj_pred = majority.predict(X_test_std)

    # model 2: multinomial logistic regression (our linear model)
    logreg = LogisticRegression(max_iter=2000, C=1.0, random_state=RANDOM_STATE)
    logreg.fit(X_train_std, Y_train)
    log_pred = logreg.predict(X_test_std)

    # model 3: random forest classifier (non-linear baseline)
    rf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=RANDOM_STATE)
    rf.fit(X_train, Y_train)
    rf_pred = rf.predict(X_test)

    print(f"Test-set results{time}")
    print(f"  Majority class : acc={accuracy_score(Y_test, maj_pred):.3f}  macroF1={f1_score(Y_test, maj_pred, average='macro'):.3f}")
    print(f"  Logistic reg   : acc={accuracy_score(Y_test, log_pred):.3f}  macroF1={f1_score(Y_test, log_pred, average='macro'):.3f}")
    print(f"  Random Forest  : acc={accuracy_score(Y_test, rf_pred):.3f}  macroF1={f1_score(Y_test, rf_pred, average='macro'):.3f}")

    LABELS = ["Strong Down", "Moderate Down", "Flat", "Moderate Up", "Strong Up"]
    print("\nPer-class report (logistic regression):")
    print(classification_report(Y_test, log_pred, labels=[0, 1, 2, 3, 4], target_names=LABELS, digits=3, zero_division=0))
    print("\nPer-class report (random forest):")
    print(classification_report(Y_test, rf_pred, labels=[0, 1, 2, 3, 4], target_names=LABELS, digits=3, zero_division=0))



In [32]:
# enter the timeframe to run analysis on
Tm = input("enter name of timeframe")
mlearn_timeframe(Tm)



Size of X_train (789, 14)
Size of y_train (789,)
Size of X_val (168, 14)
Size of y_val (168,)
Size of X_test (168, 14)
Size of y_test (168,)
Test-set resultsweekly
  Majority class : acc=0.274  macroF1=0.107
  Logistic reg   : acc=0.244  macroF1=0.160
  Random Forest  : acc=0.280  macroF1=0.267

Per-class report (logistic regression):
               precision    recall  f1-score   support

  Strong Down      0.083     0.029     0.043        35
Moderate Down      0.154     0.200     0.174        40
         Flat      0.000     0.000     0.000         0
  Moderate Up      0.310     0.565     0.400        46
    Strong Up      0.333     0.128     0.185        47

     accuracy                          0.244       168
    macro avg      0.176     0.184     0.160       168
 weighted avg      0.232     0.244     0.211       168


Per-class report (random forest):
               precision    recall  f1-score   support

  Strong Down      0.161     0.143     0.152        35
Moderate Down      

## Summary of Baseline, Logistic Regression, & Random Forest Results

Here is the consolidated summary of the classification results across all three trading timeframes, comparing the **Majority-Class Baseline (Floor)**, **Multinomial Logistic Regression**, and the **Random Forest Classifier (max_depth=6)** on the test dataset after resolving target quantile leakage:

### 1. Performance Comparison Table

| Timeframe | Train Size | Test Size | Baseline Acc (F1) | LogReg Acc (F1) | RF Acc (F1) | Best Model / Improvement vs Baseline |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **Hourly** | 30,721 | 6,830 | 25.4% (0.081) | 28.8% (0.191) | **31.6% (0.212)** | **Random Forest (+6.2% Acc, +0.131 F1)** |
| **Daily** | 3,601 | 772 | 23.4% (0.076) | 25.1% (0.173) | **27.1% (0.196)** | **Random Forest (+3.7% Acc, +0.120 F1)** |
| **Weekly** | 785 | 164 | 27.4% (0.108) | 24.4% (0.156) | **28.7% (0.279)** | **Random Forest (+1.3% Acc, +0.171 F1)** |

---

### 2. Key Insights & Analysis

1. **Leakage-Free Validation**:
   * The table above reflects target categories labeled strictly using quantile thresholds calculated from the training split. This prevents look-ahead bias and provides a true estimate of out-of-sample performance.

2. **Random Forest Outperforms Across All Timeframes**:
   * The non-linear **Random Forest** (max_depth=6) model consistently outperforms both the Majority-Class baseline and the linear Logistic Regression model in every timeframe.
   * It is particularly effective on the **weekly** data, where Logistic Regression fell below the majority baseline (24.4% vs 27.4%), while Random Forest achieved **28.7% accuracy** and a **0.279 macro F1-score** (the highest F1 achieved across any setup).

3. **The Value of Non-Linearity in Financial Time Series**:
   * Financial markets exhibit complex, non-linear relationships that simple linear models cannot capture. While Logistic Regression tends to completely ignore minority classes like **Class 2 (Flat)** and **Class 1 (Moderate Down)** (returning 0% recall), Random Forest's decision boundaries allow it to pick up more granular signals and start resolving predictions across classes, leading to significantly higher Macro F1-scores.

4. **Persistent Class Imbalance Challenges**:
   * Even with Random Forest, Class 2 (Flat) remains difficult to predict due to its extremely low support (e.g. only 139 flat hours out of 6,830 test hours). To address this in future steps, we may need to explore class-weight balancing, SMOTE, or adjusting the baseline threshold parameters to increase representation of the flat class.

## Idea: Weekly data should be the most immune to shocks generated during data trading
Particularly for semiconductor stocks like MU - which are historically cyclical but in the recent AI boom have experienced sharp volatility. So my next step should be to supplement the weekly dataset generated by Alpaca with Yahoo Finance - which allows me to pull historical data (upto 30-40 years). I want to see if supplementing the dataset helps with accuracy on the weekly dataset.

In [33]:
import yfinance as yf
# Define your list of tickers and date range
start_date = "1986-06-20"
end_date = "2016-06-20"

# Download weekly data
data = yf.download(
    tickers=MULTI_SYMBOL, 
    start=start_date, 
    end=end_date, 
    interval="1d"
)

# Preview the Close prices
data_df = pd.DataFrame(data)
data_df_long = data_df.stack(level='Ticker')
data_df_long.reset_index(inplace = True)

# Rename columns to match Alpaca's lowercase standard
data_df_long = data_df_long.rename(columns={
    'Date': 'timestamp',
    'Ticker': 'symbol',
    'Open': 'open',
    'High': 'high',
    'Low': 'low',
    'Close': 'close',
    'Volume': 'volume'
})
# Sort chronologically by symbol and timestamp (crucial for rolling indicators)
data_df_long = data_df_long.sort_values(by=['symbol', 'timestamp']).reset_index(drop=True)
# Calculate Next_Return (target return for the next period)
data_df_long['Next_Return'] = data_df_long.groupby('symbol')['close'].transform(lambda x: x.pct_change().shift(-1))


[*********************100%***********************]  4 of 4 completed


In [ ]:
# creating fetures using feature_create
weekly_data_feature_yf = feature_create(data_df_long, timeframe="daily").create_all_features()
#drop rows during warmup phase
weekly_data_feature_yf.df.dropna(inplace=True)

#Train/Val/test splitting
X_train_yf,Y_train_yf,X_val_yf,Y_val_yf,X_test_yf,Y_test_yf = preprocess_and_split(weekly_data_feature_yf.df)

# Standardizing the features using scaler fit ONLY on training set
sc_x = StandardScaler()
X_train_std_yf = sc_x.fit_transform(X_train_yf)
X_val_std_yf = sc_x.transform(X_val_yf)      
X_test_std_yf = sc_x.transform(X_test_yf)

#Joining with Alpaca daily data
X_train_std_combine = np.concat([X_train_std,X_train_std_yf],axis=1)
X_val_std_combine = np.concat([X_val_std,X_val_std_yf],axis=1)
X_test_std_combine = np.concat([X_test_std,X_test_std_yf],axis=1)

Y_train_combine = np.concat([Y_train,Y_train_yf],axis=1)
Y_val_combine = np.concat([Y_val,Y_val_yf],axis=1)
Y_test_combine = np.concat([Y_test,Y_test_yf],axis=1)


### Created a separate notebook to isolate Yfinance and look at Yfinance in isolation to Alpaca